# Lab 7: Naive Bayes — From Scratch to Bayes Optimal
### (Student Worksheet)

**Objective:** Classify sentences from biomedical abstracts into their rhetorical role
(BACKGROUND, OBJECTIVE, METHODS, RESULTS, CONCLUSIONS). You will implement Multinomial Naive
Bayes yourself in NumPy, **verify it against scikit-learn**, and then explore how smoothing,
feature representation and model combination change what the classifier can do.

**Dataset:** PubMed 20k RCT — 180,040 labelled training sentences.

**How this notebook works:** Sections 1-2 are fully worked — run them as-is. From Section 3
onward, cells marked **`# TODO`** give instructions but no solution code. Several TODOs name an
**exact variable or method**, because later cells depend on that name. Work through the
sections in order.

---
## 1. Setup & Your Assignment

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)

plt.rcParams["figure.dpi"] = 110

Set `STUDENT_ID` to **your own SRN**. Its last three digits set the size of the training
subsample used in Part C, so your Part C numbers will differ based on SRN.

In [ ]:
STUDENT_ID = "PES1UG23CS999"   # <-- CHANGE THIS TO YOUR SRN

last_three = int(STUDENT_ID[-3:])
SAMPLE_SIZE = 10_000 + last_three
SEED = 42

print(f"Student ID       : {STUDENT_ID}")
print(f"Part C subsample : {SAMPLE_SIZE:,} training sentences")

---
## 2. Load the Data


Upload `train.txt`, `dev.txt` and `test.txt` from the PubMed 20k RCT dataset next to this
notebook first.

| Split | Purpose |
|---|---|
| **train** | Fits the model — counts the words. |
| **dev** | Chooses hyperparameters (`alpha`, n-gram range). |
| **test** | Scored **once**, at the end. Never used to make a choice. |

In [ ]:
def load_pubmed_rct_file(filepath):
    """Read one PubMed 20k RCT .txt file into a DataFrame of (label, sentence)."""
    labels, sentences = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue                      # skips '###<id>' lines and blanks
            label, sent = line.split("\t", maxsplit=1)
            labels.append(label)
            sentences.append(sent)
    return pd.DataFrame({"label": labels, "sentence": sentences})


dir_path = "./"
train_df = load_pubmed_rct_file(os.path.join(dir_path, "train.txt"))
dev_df   = load_pubmed_rct_file(os.path.join(dir_path, "dev.txt"))
test_df  = load_pubmed_rct_file(os.path.join(dir_path, "test.txt"))

X_train, y_train = train_df["sentence"], train_df["label"]
X_dev,   y_dev   = dev_df["sentence"],   dev_df["label"]
X_test,  y_test  = test_df["sentence"],  test_df["label"]
target_names = sorted(y_train.unique())

print(f"Train : {len(train_df):,}")
print(f"Dev   : {len(dev_df):,}")
print(f"Test  : {len(test_df):,}")
print(f"Classes: {target_names}")

In [ ]:
dist = y_train.value_counts(normalize=True).reindex(target_names)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(dist.index, dist.values, color="#4c72b0")
ax.set_ylabel("share of training sentences")
ax.set_title("Class distribution")
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

print(dist.round(3).to_string())
print(f"\nAlways-predict-'{dist.idxmax()}' accuracy would be {dist.max():.3f}")
print("Two example sentences:")
for lab in target_names[:2]:
    print(f"  [{lab}] {train_df[train_df.label == lab].sentence.iloc[0][:90]}...")

**TODO (reflection):** The classes are far from balanced. Using the number printed above,
explain why plain **accuracy** is a weak way to score this task, and what **macro-averaged F1**
(which averages the per-class F1 scores, weighting every class equally) measures instead.
Which class do you expect to be hardest, and why?

*Your answer:*

---
## 3. Multinomial Naive Bayes, From Scratch

### Concept

Naive Bayes picks the class with the highest posterior probability given the words in the
sentence. By Bayes' rule, and dropping the evidence term (the same for every class):

$$\hat{c} = \arg\max_{c}\; P(c) \prod_{i} P(w_i \mid c)^{x_i}$$

where $x_i$ is how many times word $i$ appears. The "naive" part is that product: it assumes
words are **conditionally independent** given the class, which is plainly false for language
("randomized" and "controlled" are not independent) but works well anyway.

**Everything happens in log space.** A sentence of 25 words multiplies 25 probabilities, each
perhaps $10^{-5}$ — the result underflows to exactly 0.0 in floating point, and every class
ties at zero. Taking logs turns the product into a sum, which is numerically stable and faster:

$$\log P(c) + \sum_i x_i \log P(w_i \mid c)$$

**Laplace (additive) smoothing.** If a word never appears in class $c$ in training, its
probability is 0, the product is 0, and that class is ruled out by a single unseen word — no
matter how strongly the other 24 words support it. Smoothing adds a pseudo-count $\alpha$ to
every word:

$$P(w_i \mid c) = \frac{\text{count}(w_i, c) + \alpha}{\sum_j \text{count}(w_j, c) + \alpha V}$$

where $V$ is the vocabulary size. The $\alpha V$ in the denominator keeps it a valid
probability distribution. Section 6 asks you to tune $\alpha$.



In [ ]:
class NaiveBayesClassifier:
    """Multinomial Naive Bayes with Laplace smoothing, implemented from scratch."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        """X: sparse count matrix (n_samples, vocab_size).  y: array-like of labels."""
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.vocabulary_size_ = X.shape[1]
        n_samples = X.shape[0]

        log_priors, log_likelihoods = [], []
        for c in self.classes_:
            X_c = X[y == c]                   # rows belonging to this class

            # ---- TODO (1): log prior for class c ---------------------------
            #      P(c) = (rows in class c) / (total rows), then take log.
            #      Append it to log_priors.

            # counts[j] = how many times word j occurs across all of class c
            counts = np.asarray(X_c.sum(axis=0)).ravel()

            # ---- TODO (2): smoothed log likelihood for class c -------------
            #      numerator   = counts + alpha                 (shape: vocab_size)
            #      denominator = counts.sum() + alpha * vocab_size   (a scalar)
            #      Append log(numerator / denominator) to log_likelihoods.

            pass                              # <- remove once (1) and (2) are written

        # ---- TODO (3): stack the lists into arrays -------------------------
        #      self.class_log_prior_  -> np.array(log_priors)       (n_classes,)
        #      self.feature_log_prob_ -> np.vstack(log_likelihoods) (n_classes, vocab_size)

        return self

    def predict_log_scores(self, X):
        """Un-normalised log posterior for every class. Shape (n_samples, n_classes)."""
        # ---- TODO (4): one matrix multiply, no loops -----------------------
        #      score[i, c] = log P(c) + sum_j  X[i, j] * log P(w_j | c)
        #      That is exactly:  X @ self.feature_log_prob_.T + self.class_log_prior_
        #      (the prior is a row of length n_classes and broadcasts down the rows)
        pass

    def predict(self, X):
        """Return the predicted label for every row of X."""
        # ---- TODO (5): argmax across the class axis, then map index -> label
        #      Careful: np.argmax gives you a column INDEX. Convert it to a label
        #      with self.classes_[...], or every prediction will be an integer.
        pass


---
## 4. Build the Features & Verify Your Implementation

First turn sentences into count vectors. `CountVectorizer` is fitted on the **training split
only** — fitting it on everything would let the vocabulary be shaped by sentences the model is
not supposed to have seen.

`min_df=2` drops words appearing in only one sentence across 180,000. Those are typos and
one-off technical terms; keeping them inflates the vocabulary (and therefore the $\alpha V$
smoothing term) without adding signal.

In [ ]:
# TODO: build the count features.
#
#   1. Create `count_vectorizer = CountVectorizer(...)` with:
#          lowercase=True, strip_accents="unicode", stop_words="english",
#          ngram_range=(1, 1), min_df=2
#   2. Fit it on X_train and transform in one call -> X_train_counts
#   3. Transform (do NOT re-fit) X_dev and X_test -> X_dev_counts, X_test_counts
#
# Use these exact names; the rest of the notebook depends on them.

# count_vectorizer = ...
# X_train_counts = ...
# X_dev_counts   = ...
# X_test_counts  = ...


### The verification check 



The reason to check this way: **a wrong Naive Bayes still classifies.** Forget the $\alpha V$
term in the denominator and your probabilities no longer sum to one — but they are still
monotone in the counts, so the argmax is often unchanged and accuracy barely moves. Drop the
log prior entirely and you lose a couple of points, which looks like a modelling choice rather
than a bug. Neither failure announces itself.

In [ ]:
_mine = NaiveBayesClassifier(alpha=1.0).fit(X_train_counts, y_train)
_skl  = MultinomialNB(alpha=1.0).fit(X_train_counts, y_train)

assert hasattr(_mine, "class_log_prior_"), "fit() never set class_log_prior_"
assert hasattr(_mine, "feature_log_prob_"), "fit() never set feature_log_prob_"
assert _mine.feature_log_prob_.shape == _skl.feature_log_prob_.shape, (
    f"feature_log_prob_ is {_mine.feature_log_prob_.shape}, "
    f"expected {_skl.feature_log_prob_.shape} (n_classes, vocab_size)")

prior_err = np.abs(_mine.class_log_prior_ - _skl.class_log_prior_).max()
lik_err   = np.abs(_mine.feature_log_prob_ - _skl.feature_log_prob_).max()
print(f"Max difference in log priors      : {prior_err:.3e}")
print(f"Max difference in log likelihoods : {lik_err:.3e}")

assert prior_err < 1e-10, (
    "LOG PRIORS DO NOT MATCH. P(c) is the share of training ROWS in class c "
    "(X_c.shape[0] / n_samples) — not the share of word counts.")
assert lik_err < 1e-10, (
    "LOG LIKELIHOODS DO NOT MATCH. Most likely the denominator: it must be "
    "counts.sum() + alpha * vocabulary_size, and the alpha*V term is easy to omit.")

_agree = (_mine.predict(X_test_counts) == _skl.predict(X_test_counts)).mean()
print(f"Prediction agreement on test      : {_agree:.4%}")
assert _agree > 0.999, "Parameters match but predictions do not — check predict()."

print("\nSection 4 checks passed — your implementation is correct.")

**TODO (reflection):** Suppose you had forgotten the `alpha * vocabulary_size` term in the
denominator. The check above would fail, but would the **accuracy** have dropped noticeably?
Reason about what that term does to the scores of each class, and explain why matching a
reference implementation is a stronger test than "the accuracy looks about right".

*Your answer:*

---
## 5. Baseline Evaluation

Train your classifier on the count features and score it on the test set.

In [ ]:
RESULTS = []    # every experiment appends one row here


def log_result(name, y_true, y_pred, note="", elapsed=None):
    """Score a set of predictions, print a summary, and record it in RESULTS."""
    row = {
        "Experiment": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro"), 4),
        "Time (s)": None if elapsed is None else round(elapsed, 1),
        "Observations": note,
    }
    RESULTS.append(row)
    print(f"  {name}: accuracy {row['Accuracy']:.4f} | macro-F1 {row['Macro F1']:.4f}")
    return row


def plot_confusion(y_true, y_pred, title):
    """Row-normalised confusion matrix: each row shows where that true class went."""
    cm = confusion_matrix(y_true, y_pred, labels=target_names)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(6.5, 5))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names,
                vmin=0, vmax=1, cbar_kws={"label": "share of true class"})
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(title)
    plt.tight_layout(); plt.show()


print("Helpers ready.")

In [ ]:
# TODO: train and evaluate the baseline.
#
#   1. nb_model = NaiveBayesClassifier(alpha=1.0), fitted on X_train_counts / y_train
#   2. y_test_pred = nb_model.predict(X_test_counts)
#   3. log_result("Custom NB (counts, alpha=1)", y_test, y_test_pred,
#                 note="<your one-line observation>", elapsed=<seconds>)
#   4. print(classification_report(y_test, y_test_pred, target_names=target_names))
#   5. plot_confusion(y_test, y_test_pred, "Custom NB — counts")
#
# Use the names `nb_model` and `y_test_pred`.

# ...your code here...


**TODO (reflection):** Look at the row-normalised confusion matrix. Name the two largest
off-diagonal confusions and explain, in terms of the words those classes share, why Naive
Bayes makes them. Does the pattern match what you predicted in Section 2?

*Your answer:*

---
# Part B — Smoothing and Features

## 6. Experiment 1 — The Smoothing Parameter

### Concept

$\alpha$ controls how much probability mass is reserved for words a class has never produced.
Small $\alpha$ trusts the training counts almost completely, so a single unseen word can
dominate the score; large $\alpha$ flattens the word distributions toward uniform, washing out
the differences between classes. Somewhere in between is a best value, and the **dev set** is
where you find it — using the test set to pick $\alpha$ would make the final score a
self-fulfilling one.

In [ ]:
# TODO: sweep alpha on the DEV set.
#
#   For alpha in [0.01, 0.1, 0.5, 1.0, 2.0]:
#       fit a NaiveBayesClassifier on X_train_counts / y_train
#       predict on X_dev_counts and compute macro-F1 against y_dev
#
#   Store the results in `alpha_scores` (a dict: alpha -> dev macro-F1),
#   pick the winner into `best_alpha`, then plot dev F1 against alpha
#   (a log-scaled x-axis reads best here: plt.xscale("log")).
#
#   Finally, refit at best_alpha, predict on TEST once, and log_result(...)
#   under the name f"Custom NB (counts, tuned alpha={best_alpha})".

# alpha_scores = {}
# ...your code here...


**TODO (reflection):** `alpha=0.01` should score clearly worse than `alpha=1.0`, even though
it fits the training counts more faithfully. Explain what goes wrong when smoothing is too
weak. Then explain why the whole sweep had to run on the dev set — what exactly would be wrong
with the final test number if you had picked `alpha` by testing each value on the test set?

*Your answer:*

---
## 7. Experiment 2 — Counts vs TF-IDF

### Concept

Raw counts treat every occurrence of every word as equally informative. **TF-IDF** rescales
each count by how rare the word is across the corpus, so "patients" — which appears everywhere
and separates nothing — is damped, while a word concentrated in one class keeps its weight.

It is the standard default for text classification, so the interesting question is whether it
actually helps *here*. Note that TF-IDF produces fractional values, not counts, so the
multinomial model is no longer being fed the kind of data its derivation assumes; it still
works, but the justification is now empirical rather than principled.


In [ ]:
# TODO: compare counts against TF-IDF, then tune the TF-IDF model.
#
#   Step 1 — a default TF-IDF model:
#       pipeline = Pipeline([("tfidf", TfidfVectorizer(lowercase=True,
#                                 strip_accents="unicode", stop_words="english", min_df=2)),
#                            ("nb", MultinomialNB())])
#       Fit on (X_train, y_train), predict X_test, and log_result(...).
#       Compare it against your Section 6 row before reading on.
#
#   Step 2 — tune it on the DEV set with GridSearchCV:
#       param_grid = {"tfidf__ngram_range": [(1, 1), (1, 2)],
#                     "nb__alpha": [0.1, 0.5, 1.0]}
#       grid = GridSearchCV(pipeline, param_grid, cv=3, scoring="f1_macro", n_jobs=-1)
#       Fit the grid on (X_dev, y_dev), then print grid.best_params_ and grid.best_score_.
#
#   Step 3 — refit the winning settings on the FULL training set, predict on
#       test once, and log_result(...).  (The grid only ever saw dev.)

# ...your code here...


**TODO (reflection):** The **default** TF-IDF model should score *worse* than your plain
count model from Section 6, which is the opposite of what its reputation suggests — and then
tuning should push it past both.

(a) Why does untuned TF-IDF lose to raw counts here? Think about what TF-IDF values look like
compared to counts, and what `alpha=1.0` does to each.

(b) What did adding bigrams buy, and what does it cost?

*Your answer:*

---
# Part C — The Bayes Optimal Classifier

## 8. Combining Hypotheses by Their Posterior

### Concept

Everything so far picks a single model and uses it. The **Bayes Optimal Classifier** says that
is not the best you can do. Rather than committing to one hypothesis $h$, it averages the
predictions of *every* hypothesis, each weighted by how probable that hypothesis is given the
data:

$$P(c \mid x, D) = \sum_{h \in H} P(c \mid x, h)\, P(h \mid D)$$

No classifier using the same hypothesis space can do better on average — which is what
"optimal" means here. The catch is that $H$ is the space of *all* hypotheses, which is
infinite, so the true BOC is not computable. What we build instead is an approximation:
take a handful of trained models as a stand-in for $H$, estimate each one's posterior weight
$P(h \mid D)$, and combine their predicted probabilities by **soft voting**.

**Estimating $P(h \mid D)$.** With equal priors over the four models, $P(h \mid D)$ is
proportional to the likelihood of held-out data under $h$. So: hold out part of the training
sample, compute each model's average log-likelihood of the true labels on it, and turn those
into weights with a softmax. Section 8 also asks you to compute the weights the *exact* way —
with the summed log-likelihood instead of the mean — and compare.

Part C uses a subsample of `SAMPLE_SIZE` training sentences (from your SRN), so these models
are weaker than the full-data models in Parts A and B. Compare them with each other, not with
earlier rows.

In [ ]:
tfidf_params = dict(lowercase=True, strip_accents="unicode", stop_words="english",
                    ngram_range=(1, 1), min_df=5)

hypotheses = {
    "NaiveBayes": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                            ("clf", MultinomialNB(alpha=0.5))]),
    "LogisticRegression": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                                    ("clf", LogisticRegression(max_iter=1000,
                                                               random_state=SEED))]),
    "DecisionTree": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                              ("clf", DecisionTreeClassifier(max_depth=20,
                                                             random_state=SEED))]),
    "RandomForest": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                              ("clf", RandomForestClassifier(n_estimators=60, max_depth=25,
                                                             random_state=SEED, n_jobs=-1))]),
}

X_sample = X_train[:SAMPLE_SIZE]
y_sample = y_train[:SAMPLE_SIZE]
print(f"Part C training subsample: {len(X_sample):,} sentences")
print(f"Hypothesis space H: {list(hypotheses)}")

In [ ]:
# TODO: estimate the posterior weight of each hypothesis.
#
#   1. Split the subsample into a fitting part and a held-out part:
#          X_fit, X_held, y_fit, y_held = train_test_split(
#              X_sample, y_sample, test_size=0.25, random_state=SEED, stratify=y_sample)
#
#   2. For each (name, model) in hypotheses.items():
#          fit the model on (X_fit, y_fit)
#          proba = model.predict_proba(X_held)
#          Clip it away from zero first — np.clip(proba, 1e-12, None) — or a single
#          confident mistake gives log(0) = -inf and destroys the whole weight.
#          Pick out the probability the model gave to the TRUE label of each row:
#              col = [list(model.classes_).index(label) for label in y_held]
#              true_p = proba[np.arange(len(y_held)), col]
#          Store the MEAN of log(true_p) in `mean_log_lik[name]`.
#
#   3. Turn the mean log-likelihoods into weights with a softmax:
#          lls = np.array([mean_log_lik[n] for n in hypotheses])
#          w = np.exp(lls - lls.max());  posterior_weights = w / w.sum()
#      (subtracting the max before exponentiating is what keeps it from overflowing)
#
#   Print each model's mean log-likelihood, its accuracy on the held-out part,
#   and its final weight.

# mean_log_lik = {}
# ...your code here...


In [ ]:
# TODO: build and evaluate the Bayes Optimal Classifier.
#
#   boc = VotingClassifier(estimators=list(hypotheses.items()), voting="soft",
#                          weights=list(posterior_weights), n_jobs=-1)
#   Fit it on the FULL subsample (X_sample, y_sample) — not the 75% split — then
#   predict on X_test and log_result(...).
#
#   Then, for comparison, fit each individual hypothesis on (X_sample, y_sample),
#   score it on the test set, and log_result each one. You need those rows to
#   answer the reflection below.

# ...your code here...


**TODO (reflection):** Two things here, and the second is the important one.

(a) Compare the BOC's macro-F1 against each individual hypothesis. Did the ensemble beat all of
its members? If not, look at the weights — which models received substantial weight, and how
did they perform alone?

(b) The two weight calculations give very different answers: the mean-log-likelihood softmax
spreads weight across the models, while the summed version puts essentially **all** of it on
one. The summed version is the mathematically correct $P(h \mid D)$. Explain why it collapses,
what that collapse implies about the Bayes Optimal Classifier when $H$ contains only a handful
of hypotheses, and which of the two would have scored better here.

*Your answer:*

---
## 9. Results Table

In [ ]:
# TODO: assemble the results.
#       results_df = pd.DataFrame(RESULTS).sort_values("Macro F1", ascending=False)
#       Display it and save it with results_df.to_csv("lab7_results.csv", index=False)
#
#       Make sure every row has a non-empty "Observations" entry before you submit.

# ...your code here...


---
## 10. Reflection

Answer each in 3-5 sentences, referring to **your own numbers**.

1. **Naive Bayes assumes words are conditionally independent given the class, which is plainly
   false for real language.** Give a concrete example of two words in this dataset that are
   obviously dependent, explain what the model does wrong because of it, and suggest why the
   classifier still works reasonably well despite the assumption being violated.

   *Your answer:*

2. **Rank the three things you varied — smoothing strength, feature representation, and model
   combination — by how much each changed your macro-F1.** Which gave the best return for the
   effort involved?

   *Your answer:*

3. **The dev set was used to choose `alpha` and the n-gram range; the test set was scored only
   after those choices were fixed.** Explain what would have been wrong with the reported test
   figure if you had used the test set for tuning, and why the problem is not fixed by the test
   set being large.

   *Your answer:*

4. **You verified your implementation against scikit-learn before measuring its accuracy.**
   Describe one bug that check would catch which a reasonable-looking accuracy score would not,
   and explain why that kind of silent error is harder to find in a classifier than a crash.

   *Your answer:*